<a href="https://colab.research.google.com/github/dharani-15-star/git-tuts/blob/master/fcc_sms_text_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# import libraries
import tensorflow as tf
import pandas as pd
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt

print(tf.__version__)

In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

In [ ]:
# Load the training and validation data
train_data = pd.read_csv(train_file_path, sep="\t", header=None, names=["label", "message"])
test_data = pd.read_csv(test_file_path, sep="\t", header=None, names=["label", "message"])

# Convert labels to numbers: ham = 0, spam = 1
train_data["label"] = train_data["label"].map({"ham": 0, "spam": 1})
test_data["label"] = test_data["label"].map({"ham": 0, "spam": 1})

# Separate messages and labels
train_messages = train_data["message"].values
train_labels = train_data["label"].values

test_messages = test_data["message"].values
test_labels = test_data["label"].values

print("Training samples:", len(train_messages))
print("Validation samples:", len(test_messages))

In [ ]:
# Text preprocessing
vectorizer = keras.layers.TextVectorization(
    max_tokens=20000,
    output_mode="tf-idf",
    ngrams=2
)

# Learn vocabulary from training messages
vectorizer.adapt(train_messages)

# Build the neural network
model = keras.Sequential([
    vectorizer,
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(1, activation="sigmoid")
])

# Compile the model
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Train the model
history = model.fit(
    train_messages,
    train_labels,
    epochs=15,
    batch_size=32,
    validation_data=(test_messages, test_labels),
    verbose=1
)

# Evaluate the model
loss, accuracy = model.evaluate(
    test_messages,
    test_labels,
    verbose=0
)

print("Validation accuracy:", accuracy)

In [ ]:
# function to predict messages based on model
# (should return list containing prediction and label, ex. [0.008318834938108921, 'ham'])
def predict_message(pred_text):

  prediction = float(
      model.predict(tf.constant([pred_text]), verbose=0)[0][0]
  )

  if prediction >= 0.5:
    label = "spam"
  else:
    label = "ham"

  prediction = [prediction, label]

  return (prediction)

pred_text = "how are you doing today?"

prediction = predict_message(pred_text)
print(prediction)

In [ ]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()
